# In-class Session: Train and Test ML/DL Models

AI for Materials Science — Hands-on session 5

Can we predict an electrode's average voltage from its chemical composition? Today we use data from the Materials Project to try four models: linear regression, random forest, gradient boosting and a neural network.

The input is the fully lithiated formula, such as `Li2FePO4F`. We convert each formula into 145 numerical features. The target is `average_voltage`, reported in volts relative to Li/Li⁺. The model learns the relationship between these features and the target from examples.

## Today's plan


- **0. Setup:** install the extra libraries and download the data.
- **A. Data:** inspect the electrode table and calculate composition features.
- **B. Baseline:** set aside a test set, fit a linear model and read its errors.
- **C. Tree models:** compare models with cross-validation and practise classification.
- **D. Neural network:** train an MLP and use validation loss to choose its checkpoint.
- **E. Assignment:** train a model yourself and submit a results table and a parity plot.

Sections A–D contain worked examples and two short practice tasks. Section E is the attendance assignment.

Use training data to fit the models, and cross-validation or validation data to make model choices. Use the test data to report the final performance. Choosing settings from the test results would make those results too optimistic.

Data: Materials Project insertion electrodes (Li). Jain et al., APL Mater. 1, 011002 (2013); accessed 2026-10-22. Full references are at the end.


## 0. Setup


### 0-1. Install the libraries


Run this cell before importing the libraries. It installs `matminer` and `pymatgen`, which calculate numerical features from chemical formulas. The specified versions match those used to prepare the saved features. Colab already includes the other libraries used here.


In [ ]:
!pip install -q matminer==0.10.1 pymatgen==2026.9.24

### 0-2. Download the course material


This cell copies the course repository into a folder called `MS49900-AI4M`. The notebook expects these two files inside its `Data/` folder:

- `mp_li_electrodes.csv`: electrode formulas and target properties.
- `mp_li_electrodes_magpie145.csv.gz`: saved composition features, used if feature calculation is unavailable.

If git says the destination already exists, it leaves that folder unchanged. It does not update the files. If a later cell reports a missing file, check the contents of `MS49900-AI4M/Data/`; rerunning this clone command will not add files to an existing folder.


In [ ]:
!git clone https://github.com/kwongibaek/MS49900-AI4M.git

### 0-3. Import the libraries


The next cell imports the tools used throughout the notebook. `np`, `pd` and `plt` are short names for NumPy, pandas and Matplotlib. `nn` contains PyTorch's neural-network layers.


In [ ]:
import copy, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import (HistGradientBoostingRegressor, RandomForestClassifier,
                              RandomForestRegressor)
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (ConfusionMatrixDisplay, accuracy_score, f1_score,
                             mean_absolute_error, mean_squared_error, precision_score,
                             r2_score, recall_score)
from sklearn.model_selection import (GroupKFold, GroupShuffleSplit, cross_val_predict,
                                     cross_validate)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeRegressor, plot_tree
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
try:   ## only the live featurizer in A-3 needs these two libraries
    from pymatgen.core import Composition
    from matminer.featurizers.base import MultipleFeaturizer
    from matminer.featurizers.composition import (ElementProperty, IonProperty,
                                                  Stoichiometry, ValenceOrbital)
    HAVE_MATMINER = True
except ImportError as err:   ## 0-1 failed: A-3 reads the saved features instead
    HAVE_MATMINER = False
    print("matminer/pymatgen not importable; A-3 uses the saved features.", err)

np.random.seed(0)
torch.manual_seed(0)

### 0-4. Set the file paths


`DATA_DIR` points to the downloaded data. `OUTPUT` is the folder where the notebook saves figures and your submission files; it is created if needed.

`Data folder exists: True` confirms that the folder exists. It does not check whether the two required data files are inside it. If the folder is missing, check the download in 0-2.


In [ ]:
DATA_DIR = Path("MS49900-AI4M/Data")
OUTPUT = Path("outputs")
OUTPUT.mkdir(exist_ok=True)
print("Data folder exists:", DATA_DIR.exists())

## A. The data and its features


### A-1. Load the electrode table

Each row describes one lithium insertion electrode computed by the Materials Project. The table includes its composition and its average voltage against Li/Li⁺ during lithium insertion.

`battery_formula` shows the lithium range. For example, `Li0-2FePO4F` means 0 to 2 Li atoms per formula unit. `formula_discharge` gives the fully lithiated composition, `Li2FePO4F` in this example.

The cell prints the table size as `(rows, columns)` and shows the first five rows of selected columns.


In [ ]:
df = pd.read_csv(DATA_DIR / "mp_li_electrodes.csv")
print("rows, columns:", df.shape)
cols = ["battery_formula", "formula_discharge",
        "average_voltage", "capacity_grav"]
df[cols].head()

### A-2. What one row means

- `formula_discharge`: the fully lithiated composition.
- `average_voltage`: average voltage in V vs Li/Li⁺. This is today's prediction target.
- `capacity_grav`: capacity in mAh/g. This is the alternative target for the assignment.

The voltage is calculated from DFT reaction energies relative to Li metal. The capacity depends on how much Li the host takes up per gram. These values are computed, not measured.

`describe()` shows the number of values, mean, standard deviation (`std`), minimum, quartiles and maximum. The `std` describes how widely the values vary. For voltage, it also gives a rough scale for the RMSE of a constant mean prediction. A useful model should reduce that error. Compare the capacity's maximum with its `75%` value to see its long upper tail.


In [ ]:
df[["average_voltage", "capacity_grav", "nelements"]].describe().round(2)

### A-3. From formula to 145 composition features

A model needs the same number of numerical inputs for every sample. Each `formula_discharge` becomes 145 **features** describing its elements and their proportions. For example, `MagpieData mean Electronegativity` is the composition-weighted mean electronegativity.

We use the session 2 features: stoichiometry (6), Magpie element-property statistics (132), valence-orbital fractions (4) and ionic attributes (3).

The cell computes features with matminer. If this fails or matminer is unavailable, it loads a saved copy made with the same pinned versions. The output shows the matrix size and source.

`ignore_errors=True` lets a failed calculation produce missing values (`NaN`) instead of stopping. The imputers in B-3 and C-1 can fill missing feature values.


In [ ]:
X_all, source = None, "saved copy (matminer not installed)"
if HAVE_MATMINER:
    try:   ## construction and computation are both optional
        featurizer = MultipleFeaturizer([
            Stoichiometry(), ElementProperty.from_preset("magpie"),
            ValenceOrbital(props=["frac"]), IonProperty(fast=True)])
        featurizer.set_n_jobs(1)
        comps = [Composition(f) for f in df["formula_discharge"]]
        feats = featurizer.featurize_many(comps, ignore_errors=True,
                                          pbar=False)
        X_all = pd.DataFrame(feats, index=df.index,
                             columns=featurizer.feature_labels())
        source = "computed now"
    except Exception as err:   ## either failed: use the saved copy below
        source = f"saved copy ({type(err).__name__})"
if X_all is None:   ## the saved copy of exactly the same 145 features
    saved = pd.read_csv(DATA_DIR / "mp_li_electrodes_magpie145.csv.gz")
    X_all = df[["battery_id"]].merge(saved, on="battery_id", how="left")
    X_all = X_all.drop(columns="battery_id")
FEATURE_NAMES = list(X_all.columns)
print("feature matrix:", X_all.shape, "from", source)

## B. Split, a linear baseline, three metrics


### B-1. Repeated compositions

Different host structures can have the same `formula_discharge`. Our features describe composition, so these rows have the same inputs even if their target voltages differ.

A random row split could put the same composition in both training and test sets. That would make the test less useful for judging predictions for new compositions.

`duplicated(keep=False)` marks all rows whose formula appears more than once. The output counts these rows and the number of unique formulas.


In [ ]:
repeated = int(df["formula_discharge"].duplicated(keep=False).sum())
unique = df["formula_discharge"].nunique()
print(f"{repeated} rows share their formula with another row",
      f"({unique} unique formulas in {len(df)} rows)")

### B-2. Split by composition

We treat each `formula_discharge` as one **group**. `GroupShuffleSplit` puts all rows of a group on the same side: about 80% of formulas go to training and 20% to test. The proportions of rows can differ because groups have different sizes.

`random_state=42` makes the split repeatable. The `assert` checks that no formula appears on both sides. The output shows the row counts. We keep `groups_train` so cross-validation in C can also keep formulas together.


In [ ]:
groups = df["formula_discharge"]
y_all = df["average_voltage"]
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(splitter.split(X_all, y_all, groups=groups))
X_train, X_test = X_all.iloc[train_idx], X_all.iloc[test_idx]
y_train, y_test = y_all.iloc[train_idx], y_all.iloc[test_idx]
groups_train = groups.iloc[train_idx]
leaked = set(groups_train) & set(groups.iloc[test_idx])
assert not leaked, "a formula leaked into both sets"
print("train rows:", len(X_train), " test rows:", len(X_test))

### B-3. A linear baseline

We first fit a simple model: ordinary least squares, or linear regression. It predicts voltage from a weighted sum of the features.

A **pipeline** runs three steps in order:

1. Fill missing feature values with each column's median.
2. Standardise each column to put features on comparable scales.
3. Fit the linear model.

`fit` learns all three steps from training rows only. `predict` applies the fitted steps to new rows. This keeps test values out of the medians, scaling statistics and model fitting. The output shows the pipeline's steps.


In [ ]:
linear = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                       LinearRegression())
linear.fit(X_train, y_train)

### B-4. Three metrics

Read the training and test columns side by side:

- **MAE** is the mean absolute prediction error, in volts. Lower is better.
- **RMSE** is also in volts, but gives larger errors more weight. Lower is better.
- **R²** compares squared prediction errors with a constant mean prediction. Higher is better.

R² = 1 means exact predictions. R² = 0 means the same squared error as predicting the mean true voltage of the set being scored. Negative R² means a larger error. For test R², this reference is the **test-set mean**; it defines the score and is not supplied to the model.

Much lower training error than test error can suggest overfitting, though the sets may differ in difficulty. Compare test RMSE with the rough voltage scale from A-2.

`metrics` takes true and predicted values. `scores` obtains a fitted model's predictions, then calls `metrics`. Later sections reuse both.


In [ ]:
def metrics(y_true, y_pred):
    return {"MAE": mean_absolute_error(y_true, y_pred),
            "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
            "R2": r2_score(y_true, y_pred)}

def scores(model, X, y):
    return metrics(y, model.predict(X))

pd.DataFrame({"train": scores(linear, X_train, y_train),
              "test": scores(linear, X_test, y_test)}).round(3)

### B-5. Parity plot

The x-axis shows true voltage and the y-axis shows predicted voltage. A point on the dashed diagonal is an exact prediction. Points above it are overpredictions; points below it are underpredictions.

Compare the training and test plots. Look for systematic errors, such as underpredicting high voltages, and for predictions far outside the observed voltage range.

Strongly correlated features can make ordinary least squares unstable. It may assign large coefficients with opposite signs that cancel on training rows but give extreme predictions for new compositions. This can help explain unusual test predictions; the plot alone does not establish the cause.

We define `parity` as a function so D-7 and E can reuse it.


In [ ]:
def parity(ax, y_true, y_pred, title, target="average voltage", unit="V"):
    ax.scatter(y_true, y_pred, s=10, alpha=0.6)
    lo = float(min(y_true.min(), y_pred.min()))
    hi = float(max(y_true.max(), y_pred.max()))
    ax.plot([lo, hi], [lo, hi], "k--", linewidth=1)
    ax.set_xlabel(f"true {target} ({unit})")
    ax.set_ylabel(f"predicted ({unit})")
    ax.set_title(title)

fig, axes = plt.subplots(1, 2, figsize=(9, 4.2))
parity(axes[0], y_train, linear.predict(X_train), "Linear - train")
parity(axes[1], y_test, linear.predict(X_test), "Linear - test")
fig.tight_layout()
fig.savefig(OUTPUT / "B5_parity_linear.png", dpi=150, bbox_inches="tight")
plt.show()

### B-6. How much does the test score depend on the split?

B-4 used one split of the formulas. This cell repeats the linear-model fit with ten other splits, each holding out 20% of formulas.

The printed RMSE range shows sensitivity to the choice of held-out formulas. One split can make the model look better or worse. Extreme predictions like those in B-5 can cause large changes in RMSE.

This is a diagnostic example. We keep the B-2 split for the rest of the notebook. In C, we compare models across several validation folds within that training set.

Regularisation can reduce instability by limiting model complexity. For example, ridge regression penalises large linear coefficients. Practice 2 explores regularisation in a neural network.


In [ ]:
holdout_rmse = []
for seed in range(10):   ## ten other 80/20 splits by composition
    split_s = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=seed)
    tr, te = next(split_s.split(X_all, y_all, groups=groups))
    lin_s = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                          LinearRegression())
    lin_s.fit(X_all.iloc[tr], y_all.iloc[tr])
    holdout_rmse.append(scores(lin_s, X_all.iloc[te], y_all.iloc[te])["RMSE"])
print("test RMSE on ten splits (V):", np.round(holdout_rmse, 3))
print(f"from {min(holdout_rmse):.3f} to {max(holdout_rmse):.3f} V;",
      f"the split in B-4: {scores(linear, X_test, y_test)['RMSE']:.3f} V")

## C. Random forest, gradient boosting, cross-validation


### C-1. Three models

We define three models:

- **Linear regression**: the baseline from B.
- **Random forest**: averages 200 decision trees. Each tree trains on rows drawn with replacement, called a bootstrap sample. Averaging different trees often reduces the variability of a single deep tree.
- **Histogram gradient boosting**: adds trees in sequence to reduce remaining prediction errors. It groups feature values into bins to search for splits faster, and handles missing feature values directly.

Trees split rows using thresholds on features. They do not need standardisation, so their pipelines have no `StandardScaler`.

`cross_validate` expects higher scores to be better, so it returns negative MAE and RMSE. Our `cv_scores` function restores positive errors and reports fold averages plus the RMSE standard deviation. This cell defines the models and function; it does not train them.


In [ ]:
models = {
    "Linear": make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                            LinearRegression()),
    "RandomForest": make_pipeline(SimpleImputer(strategy="median"),
                                  RandomForestRegressor(n_estimators=200,
                                                        n_jobs=-1, random_state=0)),
    "HistGradientBoosting": HistGradientBoostingRegressor(random_state=0),
}
scoring = {"MAE": "neg_mean_absolute_error",
           "RMSE": "neg_root_mean_squared_error", "R2": "r2"}

def cv_scores(model, cv):
    res = cross_validate(model, X_train, y_train, groups=groups_train,
                         cv=cv, scoring=scoring)
    return {"CV MAE": -res["test_MAE"].mean(), "CV RMSE": -res["test_RMSE"].mean(),
            "CV RMSE std": res["test_RMSE"].std(), "CV R2": res["test_R2"].mean()}

### C-2. Five-fold cross-validation on the training set

We divide the training formulas into five **folds**. Each model trains on four folds and predicts the remaining fold. We repeat this five times, so every training row receives a validation prediction from a model trained without its formula.

`GroupKFold` keeps rows with the same formula together. These fits do not use the B-2 test set.

In the table, lower `CV MAE` and `CV RMSE` are better; higher `CV R2` is better. `CV RMSE std` shows how much RMSE varies across the five folds. Use it to judge sensitivity to the validation formulas. It is not a statistical test of whether two models differ.

Use CV results to guide model choices. Repeatedly choosing models by their test scores would make the test score too optimistic. This cell fits 15 models, so it may take a while.


In [ ]:
cv = GroupKFold(n_splits=5)
rows = [{"model": name, **cv_scores(model, cv)}
        for name, model in models.items()]
cv_table = pd.DataFrame(rows).set_index("model").round(3)
cv_table

### C-2b. One tree versus a forest

A decision tree asks a yes/no question about one feature at each split. The first node is the **root**. Further splits are **interior nodes**, and final nodes are **leaves**. A leaf predicts the mean training voltage of rows that reach it.

The first cell draws at most two levels of splits. At each node, `samples` is the number of training rows and `value` is their mean voltage. At a leaf, `value` is the prediction.

The second cell compares three depth limits using the C-2 grouped folds. If added depth lowers training RMSE but raises CV RMSE, it is overfitting: fitting training data better while predicting unseen formulas worse.

Compare with the forest in C-2. Averaging many trees often improves prediction over a single deep tree.


In [ ]:
small_tree = make_pipeline(SimpleImputer(strategy="median"),
                           DecisionTreeRegressor(max_depth=2, random_state=0))
small_tree.fit(X_train, y_train)
fig, ax = plt.subplots(figsize=(12, 4.5))
plot_tree(small_tree[-1], ax=ax, filled=True, impurity=False, precision=2,
          feature_names=[n.replace("MagpieData ", "") for n in FEATURE_NAMES])
plt.show()

In [ ]:
tree_rows = []
for depth in [3, 6, None]:   ## None: no depth limit
    tree = make_pipeline(SimpleImputer(strategy="median"),
                         DecisionTreeRegressor(max_depth=depth, random_state=0))
    res = cross_validate(tree, X_train, y_train, groups=groups_train, cv=cv,
                         scoring="neg_root_mean_squared_error", return_train_score=True)
    tree_rows.append({"max_depth": str(depth), "train RMSE": -res["train_score"].mean(),
                      "CV RMSE": -res["test_score"].mean()})
pd.DataFrame(tree_rows).set_index("max_depth").round(3)

### C-3. Fit on train, score on test

Each model now trains on all B-2 training rows and predicts the held-out test rows. For the two tree models, this is their first test-set evaluation.

Compare test RMSE with CV RMSE from C-2. Similar values support the CV estimate, but a particular test set can be easier or harder than the CV folds.

`fit s` is the training time in seconds. Use it alongside accuracy when comparing models. `fitted` stores the trained models for C-4 and D-7.


In [ ]:
fitted, test_rows = {}, []
for name, model in models.items():
    t0 = time.time()
    fitted[name] = model.fit(X_train, y_train)
    fit_s = time.time() - t0   ## training only, before predicting the test rows
    test_rows.append({"model": name, **scores(fitted[name], X_test, y_test),
                      "fit s": fit_s})
test_table = pd.DataFrame(test_rows).set_index("model").round(3)
test_table

### C-4. Which features matter?

The plot shows the random forest's 15 highest feature importances. A longer bar means the forest credited that feature with a larger reduction in training error.

This measure is called **mean decrease in impurity (MDI)**. Each split receives credit for its drop in squared error. Credits are combined across the trees and normalised so that all feature importances sum to 1.

These are training-based scores, not evidence that a feature causes voltage changes. Correlated features can share or substitute for each other's credit, so a short bar does not prove a feature is useless.

A name such as `MagpieData mean CovalentRadius` identifies an element property and a summary of that property over the composition.


In [ ]:
rf = fitted["RandomForest"].named_steps["randomforestregressor"]
importance = pd.Series(rf.feature_importances_, index=FEATURE_NAMES)
importance = importance.sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(6.5, 4.5))
importance.head(15)[::-1].plot.barh(ax=ax)
ax.set_xlabel("importance"); ax.set_title("Random forest - top 15 features")
fig.tight_layout()
fig.savefig(OUTPUT / "C4_feature_importance.png", dpi=150, bbox_inches="tight")
plt.show()

**Why might covalent radius rank highly?** Voltage depends on the energy change when lithium enters the host. A radius statistic does not describe that energy directly, but it can help the model distinguish chemical families.

For example, compositions rich in O or F often have a lower mean covalent radius than compositions rich in larger elements. These families can also differ in voltage.

The next cell groups electrodes by their elements: F-containing; O-containing without F; S, Se or Te without O or F; and none of these. It reports each group's size, mean voltage and mean covalent radius. Compare the averages to see how radius and voltage vary across these groups.

This comparison shows an association, not that covalent radius causes the voltage difference. Other correlated features may describe similar information and receive different importance rankings.


In [ ]:
## The elements column lists each electrode's elements other than Li, for example "Fe-P-O-F".
elements = df["elements"].str.split("-")
anion = np.select(
    [elements.apply(lambda e: "F" in e), elements.apply(lambda e: "O" in e),
     elements.apply(lambda e: bool({"S", "Se", "Te"} & set(e)))],
    ["contains F", "contains O (no F)", "S, Se or Te (no O, F)"], default="none of these")
anion_table = pd.DataFrame({"anion": anion, "voltage": df["average_voltage"],
                            "radius": X_all["MagpieData mean CovalentRadius"]})
(anion_table.groupby("anion")
 .agg(rows=("voltage", "size"), mean_voltage=("voltage", "mean"),
      mean_covalent_radius=("radius", "mean"))
 .sort_values("mean_voltage", ascending=False).round(2))

### C-5. A yes/no question: classification metrics

We now ask whether an electrode's average voltage is at least **4.5 V**. The features and composition split stay the same. The target becomes 1 for yes and 0 for no.

`RandomForestClassifier` estimates the probability of yes. We predict yes when it reaches a chosen **cut-off**, normally 0.5.

The confusion matrix counts correct and incorrect labels. TP means a correct yes, FP a false yes, FN a missed yes and TN a correct no.

- **Accuracy**: the fraction of correct answers. Always answering no can score well if most labels are no, while finding no candidates. The report includes this baseline.
- **Precision** = TP / (TP + FP): among predicted yes labels, the fraction that are correct.
- **Recall** = TP / (TP + FN): among actual yes labels, the fraction found.
- **F1**: the harmonic mean of precision and recall. It is high only when both are high.

For screening, recall helps avoid missed candidates; precision limits unnecessary follow-up calculations. Lowering the cut-off usually finds more candidates and creates more false alarms.

The first cell chooses the cut-off with the highest F1 on out-of-fold training predictions. Each prediction comes from a model trained without that formula. The second cell fits all training rows and reports test results for both cut-offs.

Scikit-learn's confusion matrices have true labels in rows and predictions in columns. The lecture slides use the opposite arrangement, so read the axis labels.


In [ ]:
CUT_V = 4.5   ## yes = average voltage of at least 4.5 V
c_train = (y_train >= CUT_V).astype(int)
c_test = (y_test >= CUT_V).astype(int)
print(f"share of yes: train {c_train.mean():.3f}, test {c_test.mean():.3f}")

clf = make_pipeline(SimpleImputer(strategy="median"),
                    RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=0))
## Out-of-fold probabilities: each training row is predicted by a model that did not see it.
p_oof = cross_val_predict(clf, X_train, c_train, groups=groups_train, cv=cv,
                          method="predict_proba")[:, 1]
cutoffs = np.round(np.arange(0.1, 0.91, 0.05), 2)
cv_f1 = [f1_score(c_train, p_oof >= t, zero_division=0) for t in cutoffs]
best_cutoff = float(cutoffs[int(np.argmax(cv_f1))])
print(f"CV F1 at cut-off 0.5: {cv_f1[list(cutoffs).index(0.5)]:.3f};",
      f"best CV F1 {max(cv_f1):.3f} at cut-off {best_cutoff}")

In [ ]:
def class_metrics(y_true, y_pred):
    return {"accuracy": accuracy_score(y_true, y_pred),
            "precision": precision_score(y_true, y_pred, zero_division=0),
            "recall": recall_score(y_true, y_pred),
            "F1": f1_score(y_true, y_pred, zero_division=0)}

clf.fit(X_train, c_train)
p_test = clf.predict_proba(X_test)[:, 1]
answers = {"always no": np.zeros(len(c_test), dtype=int),
           "forest, cut-off 0.5": (p_test >= 0.5).astype(int),
           f"forest, CV cut-off {best_cutoff}": (p_test >= best_cutoff).astype(int)}
fig, axes = plt.subplots(1, 2, figsize=(9, 3.8))
for ax, name in zip(axes, list(answers)[1:]):
    ConfusionMatrixDisplay.from_predictions(c_test, answers[name], ax=ax, colorbar=False,
                                            display_labels=["below 4.5 V", "4.5 V or more"])
    ax.set_title(name)
fig.tight_layout()
fig.savefig(OUTPUT / "C5_confusion_matrices.png", dpi=150, bbox_inches="tight")
plt.show()
pd.DataFrame({name: class_metrics(c_test, pred) for name, pred in answers.items()}).T.round(3)

### C-6. Practice 1

Change **one** random-forest hyperparameter, then repeat the 5-fold cross-validation. Report the CV RMSE before and after. Write one sentence explaining the change in the score.

You can change:

- `n_estimators`: the number of trees. More trees take longer and can give a steadier average.
- `max_depth`: the maximum depth of each tree. A smaller value makes the trees simpler.
- `max_features`: how many features each split can consider.

Create a separate pipeline, such as `rf_practice`, so the original model remains available for comparison. Keep the other settings, including `random_state=0`, fixed. Write your code below using `cv`, `groups_train` and `cv_scores` from C-1 and C-2.


In [ ]:
## Write your code here.
## Order: copy the RandomForest pipeline with one changed parameter -> cv_scores(...) -> print both CV RMSEs -> one sentence

## D. A neural network in PyTorch


### D-1. A validation split

Set aside 20% of the training formulas for **validation**. Keep all rows with the same formula together.

We now have three sets:

- **Fit:** updates the network's weights.
- **Validation:** monitors training and selects the saved weights.
- **Test:** reports the final score in D-7.

Split first. In D-2, fit the input scaling on the fit rows only, so validation and test rows do not affect it.


In [ ]:
val_split = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=0)
fit_idx, val_idx = next(val_split.split(X_train, y_train, groups=groups_train))
X_fit, X_val = X_train.iloc[fit_idx], X_train.iloc[val_idx]
y_fit = y_train.iloc[fit_idx].to_numpy(dtype=np.float32)
y_val = y_train.iloc[val_idx].to_numpy(dtype=np.float32)
print("fit rows:", len(X_fit), " validation rows:", len(X_val))

### D-2. Scale the inputs and build data loaders

The features have different ranges. Fill missing values with medians and standardise each feature, using statistics from the fit rows only. Apply the same transformation to validation and test rows.

Convert the arrays to `float32` tensors for PyTorch. `reshape(-1, 1)` gives the targets the same `(rows, 1)` shape as the network output.

The `DataLoader` supplies up to 64 rows per mini-batch. The network updates its weights after each batch. One **epoch** is one pass through the fit rows; `shuffle=True` changes their order each epoch. `X_test_s` stores the transformed test inputs for D-7.


In [ ]:
prep = make_pipeline(SimpleImputer(strategy="median"), StandardScaler())
prep.fit(X_fit)

def to_tensor(a):
    return torch.tensor(np.asarray(a, dtype=np.float32))

X_fit_t = to_tensor(prep.transform(X_fit))
y_fit_t = to_tensor(y_fit).reshape(-1, 1)
X_val_t = to_tensor(prep.transform(X_val))
y_val_t = to_tensor(y_val).reshape(-1, 1)
X_test_s = prep.transform(X_test)
y_test_a = y_test.to_numpy(dtype=np.float32)
train_loader = DataLoader(TensorDataset(X_fit_t, y_fit_t),
                          batch_size=64, shuffle=True)
print("scaled fit matrix:", tuple(X_fit_t.shape),
      " batches per epoch:", len(train_loader))

### D-3. Build an MLP

The network has 145 inputs, hidden layers of 128 and 64 units, and one output: predicted voltage.

`nn.Linear` calculates weighted sums with a bias. `ReLU` keeps positive values and replaces negative values with zero. ReLU lets the network learn relationships that a linear model cannot.

`Dropout(p)` randomly sets some hidden values to zero during training. The default `p=0` leaves them unchanged; you can try dropout in D-6.

The network has 27,009 trainable parameters. This gives it enough flexibility to fit the training data closely, but it may overfit. Use validation loss to check.


In [ ]:
def make_mlp(n_in, dropout=0.0):
    return nn.Sequential(nn.Linear(n_in, 128), nn.ReLU(), nn.Dropout(dropout),
                         nn.Linear(128, 64), nn.ReLU(), nn.Linear(64, 1))

torch.manual_seed(0)
mlp = make_mlp(X_fit_t.shape[1])
n_params = sum(p.numel() for p in mlp.parameters())
print(mlp)
print("trainable parameters:", n_params)

### D-4. The training loop

For each mini-batch, clear the old gradients, calculate the loss, calculate gradients, and update the weights. `Adam` controls the updates; `lr=1e-3` sets the learning rate.

After each epoch, measure fit and validation MSE with the same weights in evaluation mode. Save a copy of the weights whenever validation loss reaches a new minimum. Restore that checkpoint at the end.

`model.train()` enables dropout during updates; `model.eval()` disables it during measurement. `copy.deepcopy` keeps the saved weights from changing during later updates.

The first cell defines `train_mlp`; the second trains `mlp`. It runs for at most 200 epochs and checks elapsed time after each epoch. It stops once that time exceeds 60 seconds, so the total can be slightly longer. Different machines may complete different numbers of epochs.


In [ ]:
def train_mlp(model, loader, X_val, y_val, epochs=200, lr=1e-3,
              weight_decay=0.0, max_seconds=60):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    loss_fn = nn.MSELoss()
    X_fit_all, y_fit_all = loader.dataset.tensors   ## the whole fit set, for the epoch-end loss
    history = {"train": [], "val": []}
    best = {"loss": float("inf"), "epoch": 0, "state": None}
    start = time.time()
    for epoch in range(1, epochs + 1):
        model.train()
        for xb, yb in loader:
            optimizer.zero_grad()              ## 1. forget the previous gradients
            loss = loss_fn(model(xb), yb)      ## 2. how wrong are we?
            loss.backward()                    ## 3. gradients
            optimizer.step()                   ## 4. one step downhill
        model.eval()                           ## same weights, same mode for both losses
        with torch.no_grad():
            train_loss = loss_fn(model(X_fit_all), y_fit_all).item()
            val_loss = loss_fn(model(X_val), y_val).item()
        history["train"].append(train_loss)
        history["val"].append(val_loss)
        if val_loss < best["loss"]:
            best = {"loss": val_loss, "epoch": epoch,
                    "state": copy.deepcopy(model.state_dict())}
        if time.time() - start > max_seconds:
            break
    model.load_state_dict(best["state"])       ## restore the best validation checkpoint
    history.update(epochs_completed=epoch, seconds=time.time() - start,
                   best_epoch=best["epoch"])
    return history

In [ ]:
EPOCHS, MAX_SECONDS = 200, 60
history = train_mlp(mlp, train_loader, X_val_t, y_val_t,
                    epochs=EPOCHS, max_seconds=MAX_SECONDS)
epochs_completed = history["epochs_completed"]
seconds, best_epoch = history["seconds"], history["best_epoch"]
print(f"epochs completed {epochs_completed}/{EPOCHS}",
      f"in {seconds:.1f} s; best epoch {best_epoch}")

### D-5. Loss curves

Compare the fit and validation curves. If fit loss falls while validation loss rises for several epochs, the network may be overfitting. The curves can fluctuate, and overfitting may not appear in this run.

The dotted line marks the lowest validation loss and the checkpoint restored in D-4. The y-axis shows MSE in V² on a log scale. Take its square root to get RMSE in volts.


In [ ]:
def plot_history(h, title):
    fig, ax = plt.subplots(figsize=(6, 3.6))
    ax.plot(h["train"], label="train"); ax.plot(h["val"], label="validation")
    ax.axvline(h["best_epoch"] - 1, color="gray", linestyle=":",
               label=f"best epoch {h['best_epoch']}")
    ax.set_yscale("log"); ax.set_xlabel("epoch"); ax.set_ylabel("MSE loss (V²)")
    ax.set_title(title); ax.legend()
    return fig

fig = plot_history(history, "MLP - train vs validation loss")
fig.savefig(OUTPUT / "D5_loss_curves.png", dpi=150, bbox_inches="tight")
plt.show()

### D-6. Practice 2

Compare **one** regularisation setting or a shorter training run with the original MLP:

- Dropout: use `make_mlp(n_in, dropout=0.2)`.
- Weight decay: pass `weight_decay=1e-4` to `train_mlp` to penalise large weights.
- Fewer training epochs: reduce `epochs` in `train_mlp`.

Train a separate network, such as `mlp_practice`, and save its results as `history_practice`. Keep `mlp` and `history` for the original model, which D-7 uses. Plot the new curves with `plot_history` and compare the two best validation RMSEs: the square root of the minimum validation loss in each history.

A change does not have to improve the score. A shorter run checks fewer checkpoints. Because this loop already keeps the best checkpoint, fewer epochs can save time but may miss a better checkpoint.

Call `torch.manual_seed(0)` before `make_mlp` to repeat the initial weights. Dropout uses random numbers during training, so the mini-batch order can still differ between runs. This is a single-run comparison.

Write your code below. Choose settings using validation results; leave the test set for D-7.


In [ ]:
## Write your code here.
## Order: torch.manual_seed(0) -> make_mlp(...) -> train_mlp(...) -> plot_history(...) -> best validation RMSE for both networks

### D-7. ML or DL? Four models on one test set

Add the test score of `mlp` from D-4 to the three scikit-learn scores from C-3. All four use the same test rows. The scikit-learn models fit all training rows; the MLP reserves some for validation.

Compare MAE, RMSE and R². Treat fitting time as an approximate cost: the MLP's time also includes validation checks. Use this table as the final report; do not change settings to improve test scores.

The second cell plots predicted versus actual test voltages. It compares the MLP with the tree model that had the lowest CV RMSE in C-2. Points closer to the diagonal have smaller errors.

These results describe performance on this dataset, not which method is best for every materials problem.


In [ ]:
mlp.eval()
with torch.no_grad():
    mlp_pred = mlp(to_tensor(X_test_s)).numpy().ravel()
four = test_table.copy()
four.loc["MLP (PyTorch)"] = {**metrics(y_test_a, mlp_pred),
                             "fit s": history["seconds"]}
four = four.round(3)
four

In [ ]:
trees = ["RandomForest", "HistGradientBoosting"]
best_tree = cv_table.loc[trees, "CV RMSE"].idxmin()   ## chosen by CV in C-2, not by test
fig, axes = plt.subplots(1, 2, figsize=(9, 4.2))
parity(axes[0], y_test, fitted[best_tree].predict(X_test), f"{best_tree} - test")
parity(axes[1], y_test, mlp_pred, "MLP - test")
fig.tight_layout()
fig.savefig(OUTPUT / "D7_parity_tree_vs_mlp.png", dpi=150, bbox_inches="tight")
plt.show()

### D-8. How much error do identical compositions cause?


Rows with the same formula have the same composition features, but their structures and voltages can differ. A composition-only model must give them the same prediction. For two rows at 3 V and 4 V, the prediction with the smallest squared error is their mean, 3.5 V.

The cell repeats this calculation for every formula in the **test set**, then reports the resulting RMSE as `floor`. This is the lowest RMSE possible on these particular rows when identical formulas receive identical predictions. It uses the actual test voltages to calculate each mean, so it is a reference calculation, not a model we can use for new formulas.

Compare it with D-7. A large gap means that differences within repeated formulas do not explain all the model's error. It does not show how much a new model will improve. Formulas with just one test row contribute zero to this calculation, so the number does not capture structure differences that are absent from this table.


In [ ]:
## Each test row minus the mean voltage of the test rows with the same formula.
groups_test = groups.iloc[test_idx]
spread = y_test - y_test.groupby(groups_test).transform("mean")
floor = float(np.sqrt(np.mean(spread ** 2)))
print(f"lowest test RMSE of any composition-only model: {floor:.3f} V")
print(f"best test RMSE in D-7: {four['RMSE'].min():.3f} V")

## E. Assignment (attendance)

Choose a model or switch the target to gravimetric capacity. Train it, report train and test scores, and make a test parity plot. Submit the results table and plot on KLMS.

### E-1. Set your student ID

Replace `"demo"` with your student ID. `STUDENT_ID` names your submission files and supplies the `student_id` column in your results table.


In [ ]:
## Replace this with your own student ID; it only names your files.
STUDENT_ID = "demo"

if STUDENT_ID == "demo":
    raise ValueError("Set STUDENT_ID to your own ID before running this cell.")

print("Student ID:", STUDENT_ID)

### E-2. Choose your model and target

Set `MY_MODEL` to `linear`, `random_forest`, `hist_gradient_boosting` or `mlp`. Set `MY_TARGET` to `average_voltage` (V) or `capacity_grav` (mAh/g).

Change at least one from the defaults: `random_forest` and `average_voltage`. Keep the train/test rows from B-2. For capacity, only the target changes; its few very large values can strongly affect RMSE.


In [ ]:
## MY_MODEL: linear | random_forest | hist_gradient_boosting | mlp
MY_MODEL = "random_forest"
## MY_TARGET: average_voltage | capacity_grav
MY_TARGET = "average_voltage"

MODELS = {"linear", "random_forest", "hist_gradient_boosting", "mlp"}
assert MY_MODEL in MODELS, "unknown MY_MODEL"
assert MY_TARGET in {"average_voltage", "capacity_grav"}, "bad MY_TARGET"
y_my = df[MY_TARGET]
y_my_train, y_my_test = y_my.iloc[train_idx], y_my.iloc[test_idx]
print("model:", MY_MODEL, " target:", MY_TARGET,
      " train rows:", len(y_my_train), " test rows:", len(y_my_test))

### E-3. Train, evaluate and plot

Write the code in the empty cell below. Use `X_train`, `X_test`, `y_my_train` and `y_my_test` to create:

1. `my_results`: a DataFrame with exactly these columns in this order: `student_id, model, target, split, MAE, RMSE, R2`. Include one row for `train` and one for `test`. Fill the first three columns with `STUDENT_ID`, `MY_MODEL` and `MY_TARGET`.
2. `my_fig`: a Matplotlib figure showing test predictions versus actual values. Reuse `parity` with `target=MY_TARGET` and `unit="V"` for voltage or `unit="mAh/g"` for capacity.

For a scikit-learn model, reuse C-1's definitions and fit on `X_train` and `y_my_train`. `scores(model, X, y)` returns MAE, RMSE and R2; `pd.DataFrame([...])` can combine the two result dictionaries.

For `mlp`, follow D-1 and D-2 with `y_my_train` as the target. Rebuild the fit/validation target tensors and `DataLoader` for your chosen target; the existing loader contains voltage labels. Split training rows by formula before fitting the scaler on fit rows only. Reuse `to_tensor`, `make_mlp` and `train_mlp`. After restoring the validation checkpoint, transform inputs with the fitted `prep` and predict using `eval()` and `torch.no_grad()`, as in D-7. Use `metrics(y_true, y_pred)` for the prediction arrays: `scores` requires `.predict()`, which this MLP does not have. Report full-training and test scores. Keep test rows out of model selection.


In [ ]:
## Write your code here.
## Order: build the model for MY_MODEL -> fit on X_train, y_my_train -> scores for train and test -> my_results -> my_fig

### E-4. Save your submission

Run the cell to save:

1. `hos5_<id>_results.csv`: your results table.
2. `hos5_<id>_parity.png`: your test parity plot.

In Colab, the cell downloads both files. Allow both downloads if asked. Otherwise, find them in the `outputs` folder. Upload exactly these two files to **KLMS (W10 Lab)**.

The cell checks that `my_results` and `my_fig` exist and that the table's column names and order match E-3. Check the values, both split rows and the plot yourself before submitting.


In [ ]:
## Stop with a clear message if E-3 has not been completed.
missing = [n for n in ("my_results", "my_fig") if n not in globals()]
if missing:
    raise NameError(f"Complete E-3 first. Not defined yet: {missing}")
columns = ["student_id", "model", "target", "split", "MAE", "RMSE", "R2"]
assert list(my_results.columns) == columns, f"columns must be {columns}"
safe_id = "".join(c for c in STUDENT_ID if c.isalnum() or c in "_-")
stem = OUTPUT / f"hos5_{safe_id}"
paths = [Path(f"{stem}_results.csv"), Path(f"{stem}_parity.png")]
my_results.to_csv(paths[0], index=False)
my_fig.savefig(paths[1], dpi=180, bbox_inches="tight")
for path in paths:
    print("created" if path.stat().st_size > 0 else "MISSING", path)
print(my_results.round(3).to_string(index=False))
try:   ## in Colab, download both files to your computer for the KLMS upload
    from google.colab import files
    files.download(str(paths[0])); files.download(str(paths[1]))
except ImportError:
    print("Not in Colab: download the two files from the outputs folder.")
print("Submit both files to KLMS (W10 Lab).")

## Appendix (optional). Two importances and a noise column

Run this after section C if you want to compare two feature-importance measures.

C-4's **MDI** records how much features helped splits on the training data. **Permutation importance** shuffles one test feature at a time and measures the increase in RMSE. A larger increase means the fitted model relies more on that feature for these test predictions.

The cell adds a random-noise feature, fits a new forest and plots both measures. It takes about 20 seconds. A red bar marks the noise feature if it reaches a panel's top 15. The printed lines also show its rank.

Trees can use random noise to fit training rows, so it can receive MDI credit. Its test permutation importance is usually near zero and can vary slightly above or below zero. Correlated features can also hide each other's importance.

Use these test results for inspection only. If you want to select features or change the model, measure importance on validation data and keep the test set for the final report.


In [ ]:
## One extra column of random numbers: it carries no information on the voltage.
rng = np.random.default_rng(0)
X_train_noise = X_train.assign(random_noise=rng.normal(size=len(X_train)))
X_test_noise = X_test.assign(random_noise=rng.normal(size=len(X_test)))
rf_noise = make_pipeline(SimpleImputer(strategy="median"),
                         RandomForestRegressor(n_estimators=200,
                                               n_jobs=-1, random_state=0))
rf_noise.fit(X_train_noise, y_train)

## MDI comes from training. Permutation shuffles one test column at a time.
mdi = pd.Series(
    rf_noise.named_steps["randomforestregressor"].feature_importances_,
    index=X_train_noise.columns)
perm = permutation_importance(rf_noise, X_test_noise, y_test,
                              scoring="neg_root_mean_squared_error",
                              n_repeats=3, random_state=0)
perm = pd.Series(perm.importances_mean, index=X_train_noise.columns)

fig, axes = plt.subplots(1, 2, figsize=(10, 4.2))
panels = [(mdi, "MDI (training data)"),
          (perm, "Permutation (test data): RMSE rise (V)")]
for ax, (imp, title) in zip(axes, panels):
    top = imp.sort_values(ascending=False).head(15)[::-1]
    colors = ["tab:red" if c == "random_noise" else "tab:blue"
              for c in top.index]
    ax.barh(top.index.str.replace("MagpieData ", ""), top.values,
            color=colors)
    ax.set_title(title)
fig.tight_layout()
fig.savefig(OUTPUT / "appendix_mdi_vs_permutation.png", dpi=150,
            bbox_inches="tight")
plt.show()
for name, imp in [("MDI", mdi), ("permutation", perm)]:
    rank = int(imp.rank(ascending=False, method="min")["random_noise"])
    share = 100 * imp["random_noise"] / imp.max()
    print(f"random_noise by {name}: rank {rank} of {len(imp)}, "
          f"{share:.1f}% of the top feature")

## Data sources

### Materials Project insertion electrodes
`Data/mp_li_electrodes.csv` and `Data/mp_li_electrodes_magpie145.csv.gz`. Materials Project insertion electrodes (working ion Li), accessed 2026-10-22, licensed under CC BY 4.0 (https://creativecommons.org/licenses/by/4.0/). Cite: A. Jain, S. P. Ong, G. Hautier, W. Chen, W. D. Richards, S. Dacek, S. Cholia, D. Gunter, D. Skinner, G. Ceder, K. A. Persson, *APL Materials* 1, 011002 (2013), https://doi.org/10.1063/1.4812323. Changes: rows without average_voltage or formula_discharge removed; 0 < V < 6 kept; duplicate battery_id removed; composition features added with matminer (Ward et al., *Comput. Mater. Sci.* 152, 60 (2018), https://doi.org/10.1016/j.commatsci.2018.05.018) and Magpie properties (Ward et al., *npj Comput. Mater.* 2, 16028 (2016), https://doi.org/10.1038/npjcompumats.2016.28).